In [ ]:
"""
Complete Linear Regression Implementation from Scratch
Week 9 - Supervised Learning Project

This notebook demonstrates:
1. Linear Regression from scratch using gradient descent
2. Comparison with Scikit-Learn implementation
3. Convergence visualization
4. Model evaluation and metrics
"""

# ============================================================================
# PART 1: IMPORTS AND SETUP
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Import custom modules
import sys
sys.path.append('../src')
from models import LinearRegressionScratch, LinearRegressionNormalEquation, compare_models, calculate_metrics
from preprocessing import FeatureScaler, prepare_data_for_modeling, k_fold_cross_validation

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")


# ============================================================================
# PART 2: GENERATE SAMPLE DATA
# ============================================================================

# Generate synthetic data for demonstration
n_samples = 500
X = 2 * np.random.rand(n_samples, 1)
y = 4 + 3 * X.flatten() + np.random.randn(n_samples) * 0.5

# Create DataFrame
df = pd.DataFrame({'X': X.flatten(), 'y': y})

print("\n=== Dataset Info ===")
print(f"Number of samples: {n_samples}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nStatistics:")
print(df.describe())

# Visualize data
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.5, s=30)
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Generated Dataset', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================================
# PART 3: SPLIT DATA
# ============================================================================

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")


# ============================================================================
# PART 4: TRAIN CUSTOM LINEAR REGRESSION (GRADIENT DESCENT)
# ============================================================================

print("\n" + "="*70)
print("TRAINING CUSTOM LINEAR REGRESSION (GRADIENT DESCENT)")
print("="*70)

# Create and train model
lr_scratch = LinearRegressionScratch(
    learning_rate=0.01, 
    n_iterations=1000
)

lr_scratch.fit(X_train, y_train, verbose=True)

# Get parameters
params = lr_scratch.get_params()
print(f"\n=== Learned Parameters ===")
print(f"Weight (slope): {params['weights'][0]:.4f}")
print(f"Bias (intercept): {params['bias']:.4f}")

# Evaluate
train_score = lr_scratch.score(X_train, y_train)
test_score = lr_scratch.score(X_test, y_test)

print(f"\n=== Model Performance ===")
print(f"Train R²: {train_score:.4f}")
print(f"Test R²: {test_score:.4f}")

# Plot convergence
lr_scratch.plot_cost_history()
plt.show()


# ============================================================================
# PART 5: TRAIN WITH NORMAL EQUATION
# ============================================================================

print("\n" + "="*70)
print("TRAINING LINEAR REGRESSION (NORMAL EQUATION)")
print("="*70)

lr_normal = LinearRegressionNormalEquation()
lr_normal.fit(X_train, y_train)

print(f"\n=== Learned Parameters ===")
print(f"Weight (slope): {lr_normal.weights[0]:.4f}")
print(f"Bias (intercept): {lr_normal.bias:.4f}")

train_score_normal = lr_normal.score(X_train, y_train)
test_score_normal = lr_normal.score(X_test, y_test)

print(f"\n=== Model Performance ===")
print(f"Train R²: {train_score_normal:.4f}")
print(f"Test R²: {test_score_normal:.4f}")


# ============================================================================
# PART 6: TRAIN SCIKIT-LEARN MODEL
# ============================================================================

print("\n" + "="*70)
print("TRAINING SCIKIT-LEARN LINEAR REGRESSION")
print("="*70)

# Create and train Scikit-Learn model
lr_sklearn = LinearRegression()
lr_sklearn.fit(X_train, y_train)

print(f"\n=== Learned Parameters ===")
print(f"Weight (slope): {lr_sklearn.coef_[0]:.4f}")
print(f"Bias (intercept): {lr_sklearn.intercept_:.4f}")

# Evaluate
train_score_sklearn = lr_sklearn.score(X_train, y_train)
test_score_sklearn = lr_sklearn.score(X_test, y_test)

print(f"\n=== Model Performance ===")
print(f"Train R²: {train_score_sklearn:.4f}")
print(f"Test R²: {test_score_sklearn:.4f}")


# ============================================================================
# PART 7: STATSMODELS REGRESSION (WITH STATISTICS)
# ============================================================================

print("\n" + "="*70)
print("STATSMODELS REGRESSION ANALYSIS")
print("="*70)

# Add constant for statsmodels
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# Fit model
model_sm = sm.OLS(y_train, X_train_sm).fit()

# Print detailed statistics
print(model_sm.summary())


# ============================================================================
# PART 8: COMPARE ALL MODELS
# ============================================================================

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)

# Make predictions
y_pred_scratch = lr_scratch.predict(X_test)
y_pred_normal = lr_normal.predict(X_test)
y_pred_sklearn = lr_sklearn.predict(X_test)
y_pred_sm = model_sm.predict(X_test_sm)

# Calculate metrics for all models
models_comparison = {
    'Custom (Gradient Descent)': calculate_metrics(y_test, y_pred_scratch),
    'Custom (Normal Equation)': calculate_metrics(y_test, y_pred_normal),
    'Scikit-Learn': calculate_metrics(y_test, y_pred_sklearn),
    'Statsmodels': calculate_metrics(y_test, y_pred_sm)
}

# Create comparison DataFrame
comparison_df = pd.DataFrame(models_comparison).T
print("\n=== Metrics Comparison ===")
print(comparison_df)

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

metrics = ['MSE', 'RMSE', 'MAE', 'R2']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    values = comparison_df[metric].values
    bars = ax.bar(range(len(values)), values, color=colors[idx], alpha=0.7)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(comparison_df.index, rotation=45, ha='right')
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


# ============================================================================
# PART 9: VISUALIZE PREDICTIONS
# ============================================================================

print("\n" + "="*70)
print("PREDICTION VISUALIZATIONS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: All models predictions
ax1 = axes[0, 0]
ax1.scatter(X_train, y_train, alpha=0.3, s=20, label='Train data', color='gray')
ax1.scatter(X_test, y_test, alpha=0.5, s=30, label='Test data', color='lightblue')

X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
ax1.plot(X_line, lr_scratch.predict(X_line), 'r-', linewidth=2, label='Custom (GD)', alpha=0.7)
ax1.plot(X_line, lr_sklearn.predict(X_line), 'g--', linewidth=2, label='Scikit-Learn', alpha=0.7)

ax1.set_xlabel('X', fontsize=11)
ax1.set_ylabel('y', fontsize=11)
ax1.set_title('Model Predictions Comparison', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Predictions vs Actual
ax2 = axes[0, 1]
ax2.scatter(y_test, y_pred_scratch, alpha=0.6, s=50, label='Predictions')
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
ax2.set_xlabel('Actual Values', fontsize=11)
ax2.set_ylabel('Predicted Values', fontsize=11)
ax2.set_title('Predictions vs Actual', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Residuals
ax3 = axes[1, 0]
residuals = y_test - y_pred_scratch
ax3.scatter(y_pred_scratch, residuals, alpha=0.6, s=50)
ax3.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax3.set_xlabel('Predicted Values', fontsize=11)
ax3.set_ylabel('Residuals', fontsize=11)
ax3.set_title('Residual Plot', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Residuals distribution
ax4 = axes[1, 1]
ax4.hist(residuals, bins=30, alpha=0.7, edgecolor='black')
ax4.axvline(x=0, color='r', linestyle='--', linewidth=2)
ax4.set_xlabel('Residuals', fontsize=11)
ax4.set_ylabel('Frequency', fontsize=11)
ax4.set_title('Residuals Distribution', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


# ============================================================================
# PART 10: COMPARE DIFFERENT LEARNING RATES
# ============================================================================

print("\n" + "="*70)
print("COMPARING DIFFERENT LEARNING RATES")
print("="*70)

results, fig = compare_models(
    X_train, y_train, 
    X_test, y_test,
    learning_rates=[0.001, 0.01, 0.05, 0.1],
    n_iterations=1000
)

# Print results
print("\n=== Learning Rate Comparison Results ===")
for result in results:
    print(f"\nLearning Rate: {result['learning_rate']}")
    print(f"  Train R²: {result['train_r2']:.4f}")
    print(f"  Test R²: {result['test_r2']:.4f}")
    print(f"  Final Cost: {result['final_cost']:.4f}")

plt.show()


# ============================================================================
# PART 11: CROSS-VALIDATION
# ============================================================================

print("\n" + "="*70)
print("K-FOLD CROSS-VALIDATION")
print("="*70)

# Combine train and test for cross-validation
X_full = np.vstack([X_train, X_test])
y_full = np.concatenate([y_train, y_test])

# Perform cross-validation
cv_model = LinearRegressionScratch(learning_rate=0.01, n_iterations=500)
cv_results = k_fold_cross_validation(X_full, y_full, cv_model, k=5, random_state=42)

print(f"\n=== Cross-Validation Results ===")
print(f"Mean R²: {cv_results['mean_score']:.4f} (+/- {cv_results['std_score']:.4f})")
print(f"Min R²: {cv_results['min_score']:.4f}")
print(f"Max R²: {cv_results['max_score']:.4f}")

# Visualize CV scores
plt.figure(figsize=(10, 6))
plt.bar(range(1, 6), cv_results['scores'], alpha=0.7, edgecolor='black')
plt.axhline(y=cv_results['mean_score'], color='r', linestyle='--', 
            linewidth=2, label=f"Mean: {cv_results['mean_score']:.4f}")
plt.xlabel('Fold', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.title('5-Fold Cross-Validation Scores', fontsize=14, fontweight='bold')
plt.xticks(range(1, 6))
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


# ============================================================================
# PART 12: MULTIVARIATE LINEAR REGRESSION
# ============================================================================

print("\n" + "="*70)
print("MULTIVARIATE LINEAR REGRESSION")
print("="*70)

# Generate multivariate data
n_samples = 1000
n_features = 3

X_multi = np.random.randn(n_samples, n_features)
true_weights = np.array([2.5, -1.8, 3.2])
true_bias = 4.0

y_multi = X_multi.dot(true_weights) + true_bias + np.random.randn(n_samples) * 0.5

print(f"True weights: {true_weights}")
print(f"True bias: {true_bias}")

# Split data
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

# Train custom model
lr_multi = LinearRegressionScratch(learning_rate=0.01, n_iterations=1000)
lr_multi.fit(X_train_m, y_train_m, verbose=False)

print(f"\n=== Learned Parameters ===")
print(f"Learned weights: {lr_multi.weights}")
print(f"Learned bias: {lr_multi.bias:.4f}")

# Train sklearn model
lr_sklearn_multi = LinearRegression()
lr_sklearn_multi.fit(X_train_m, y_train_m)

# Compare
print(f"\n=== Comparison ===")
print(f"Custom Model - Test R²: {lr_multi.score(X_test_m, y_test_m):.4f}")
print(f"Sklearn Model - Test R²: {lr_sklearn_multi.score(X_test_m, y_test_m):.4f}")

# Plot convergence
lr_multi.plot_cost_history()
plt.title('Multivariate Regression - Cost Convergence', fontsize=14, fontweight='bold')
plt.show()

# Compare learned vs true weights
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(n_features)
width = 0.25

ax.bar(x_pos - width, true_weights, width, label='True Weights', alpha=0.8)
ax.bar(x_pos, lr_multi.weights, width, label='Custom Model', alpha=0.8)
ax.bar(x_pos + width, lr_sklearn_multi.coef_, width, label='Sklearn Model', alpha=0.8)

ax.set_xlabel('Feature', fontsize=12)
ax.set_ylabel('Weight Value', fontsize=12)
ax.set_title('Weight Comparison: True vs Learned', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Feature {i+1}' for i in range(n_features)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


# ============================================================================
# PART 13: REGULARIZATION (L1 and L2)
# ============================================================================

print("\n" + "="*70)
print("REGULARIZATION COMPARISON")
print("="*70)

# Train models with different regularization
lr_no_reg = LinearRegressionScratch(learning_rate=0.01, n_iterations=1000)
lr_l1 = LinearRegressionScratch(learning_rate=0.01, n_iterations=1000, 
                                 regularization='l1', lambda_reg=0.1)
lr_l2 = LinearRegressionScratch(learning_rate=0.01, n_iterations=1000, 
                                 regularization='l2', lambda_reg=0.1)

lr_no_reg.fit(X_train_m, y_train_m, verbose=False)
lr_l1.fit(X_train_m, y_train_m, verbose=False)
lr_l2.fit(X_train_m, y_train_m, verbose=False)

print(f"\n=== Model Performance ===")
print(f"No Regularization - Test R²: {lr_no_reg.score(X_test_m, y_test_m):.4f}")
print(f"L1 Regularization - Test R²: {lr_l1.score(X_test_m, y_test_m):.4f}")
print(f"L2 Regularization - Test R²: {lr_l2.score(X_test_m, y_test_m):.4f}")

# Compare weights
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Weight values
ax1 = axes[0]
x_pos = np.arange(n_features)
width = 0.25

ax1.bar(x_pos - width, lr_no_reg.weights, width, label='No Regularization', alpha=0.8)
ax1.bar(x_pos, lr_l1.weights, width, label='L1 (Lasso)', alpha=0.8)
ax1.bar(x_pos + width, lr_l2.weights, width, label='L2 (Ridge)', alpha=0.8)

ax1.set_xlabel('Feature', fontsize=11)
ax1.set_ylabel('Weight Value', fontsize=11)
ax1.set_title('Weight Comparison with Regularization', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'Feature {i+1}' for i in range(n_features)])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Cost convergence
ax2 = axes[1]
ax2.plot(lr_no_reg.cost_history, label='No Regularization', linewidth=2)
ax2.plot(lr_l1.cost_history, label='L1 (Lasso)', linewidth=2)
ax2.plot(lr_l2.cost_history, label='L2 (Ridge)', linewidth=2)
ax2.set_xlabel('Iteration', fontsize=11)
ax2.set_ylabel('Cost', fontsize=11)
ax2.set_title('Cost Convergence Comparison', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*70)
print("PROJECT SUMMARY")
print("="*70)

print("""
✓ Implemented Linear Regression from scratch using gradient descent
✓ Implemented Normal Equation solution
✓ Compared with Scikit-Learn implementation
✓ Analyzed convergence and cost function
✓ Tested different learning rates
✓ Performed k-fold cross-validation
✓ Implemented multivariate regression
✓ Added L1 and L2 regularization
✓ Comprehensive visualization and metrics

Key Findings:
1. Gradient descent converges to the same solution as normal equation
2. All implementations (custom, sklearn, statsmodels) produce similar results
3. Learning rate significantly affects convergence speed
4. Regularization helps prevent overfitting in high-dimensional data
5. Cross-validation provides robust performance estimates

Next Steps:
- Try with real-world datasets (housing prices, salary prediction)
- Implement polynomial regression
- Explore advanced regularization techniques
- Build end-to-end ML pipelines
""")

print("\n Project completed successfully!")